In [0]:
import sys
from datetime import datetime

# 1. Capture parameters from the Master API call
dbutils.widgets.text("batch_config_ids", "")
dbutils.widgets.text("config_master_id", "")
dbutils.widgets.text("source_system_id", "")
# ... declare your other standard widgets ...

batch_config_ids_str = dbutils.widgets.get("batch_config_ids")
config_master_id = int(dbutils.widgets.get("config_master_id"))
source_system_id = int(dbutils.widgets.get("source_system_id"))
target_batch_ids = [int(x.strip()) for x in batch_config_ids_str.split(",")]

# 2. Initialize Config Manager & Orchestrator
config_mgr = ConfigManager(spark, target_catalog=dbutils.widgets.get("target_catalog"))
source_sys, all_tasks = config_mgr.get_active_tasks(
    config_master_id=config_master_id,
    source_system_id=source_system_id,
    pipeline_name=dbutils.widgets.get("pipeline_name")
)

orchestrator = IngestionOrchestrator(
    spark, dbutils, 
    pipeline_name=dbutils.widgets.get("pipeline_name"),
    environment=dbutils.widgets.get("environment"),
    silver_notebook_path=dbutils.widgets.get("silver_notebook_path"),
    config_mgr=config_mgr
)

# 3. Filter only the tasks assigned to this specific batch
batch_tasks = [t for t in all_tasks if t.config_id in target_batch_ids]

# The tasks are already sorted by priority in ConfigManager.get_active_tasks[cite: 3].
for task in batch_tasks:
    print(f"Executing config_id {task.config_id} (Priority: {task.priority})")
    
    # orchestrator.run() handles the coupled extract, raw write, and silver trigger[cite: 4].
    result = orchestrator.run(
        source_sys=source_sys,
        task=task,
        config_master_id=config_master_id,
        landing_volume_path=dbutils.widgets.get("landing_volume_path"),
        trigger_id=dbutils.widgets.get("job_run_id"),
        process_timestamp=datetime.utcnow()
    )
    
    if result["status"] == "FAILED":
        raise Exception(f"Extraction failed for config_id {task.config_id}. Aborting Batch.")